In [ ]:
# ============================================================
# REGRAS DA LÓGICA
# ============================================================
# ~   : negação
# &   : conjunção
# |   : disjunção
# ->  : implicação
# <-> : bicondicional
# Toda operação binária deve estar entre parênteses
# ============================================================

import pandas as pd

def operador_principal(expr):

    nivel = 0

    for i in range(len(expr)):

        if expr[i] == '(':
            nivel += 1

        elif expr[i] == ')':
            nivel -= 1

        elif nivel == 0:

            if expr[i:i+3] == '<->':
                return '<->', i

            if expr[i:i+2] == '->':
                return '->', i

            if expr[i] == '&':
                return '&', i

            if expr[i] == '|':
                return '|', i

    return None, None
def avaliar(expr, valores):

    expr = remover_parenteses_externos(expr)

    if len(expr) == 1:
        return valores[expr]

    if expr[0] == '~':
        return not avaliar(expr[1:], valores)

    op, pos = operador_principal(expr)

    if op is None:
        raise Exception(f"Operador principal não encontrado em {expr}")

    esquerda = expr[:pos]
    direita = expr[pos + len(op):]

    a = avaliar(esquerda, valores)
    b = avaliar(direita, valores)

    if op == '&':
        return a and b

    elif op == '|':
        return a or b

    elif op == '->':
        return (not a) or b

    elif op == '<->':
        return a == b

def remover_parenteses_externos(expr):
    # Remove parênteses redundantes externos
    while True:
        if len(expr) < 2 or expr[0] != "(" or expr[-1] != ")":
            return expr

        nivel = 0
        pode_remover = True

        for i in range(len(expr)):
            if expr[i] == "(":
                nivel += 1
            elif expr[i] == ")":
                nivel -= 1

            # se fechar antes do final, não pode remover
            if nivel == 0 and i != len(expr) - 1:
                pode_remover = False
                break

        if pode_remover:
            expr = expr[1:-1] # Faz o slicing para tirar o primeiro e o último dígito, que são os parênteses
        else:
            return expr


def encontrar_relacoes(expr,coluna_legenda): # Coluna legenda é a lista onde todas as subexpressões encontradasm serão armazenadas
    # Extrai subexpressões entre parênteses
    pilha = [] # A pilha guarda as posições dos parênteses de abertura

    for i in range(len(expr)): # i é o índice do caractere atual
        if expr[i] == "(":
            pilha.append(i) # Guarda o índice na pilha, e continu aprocurando por outros parênteses do tipo "("

        elif expr[i] == ")": #  Retira o último parêntese da pilha para evitar experessões inválidas, como "((A&B)"
            inicio = pilha.pop() # Inicio é o índice (a posição) do parêntese de abertura que acabou de ser encontrado

            # ignora expressão principal inteira
            if inicio == 0 and i == len(expr) - 1:
                continue

            sub = expr[inicio:i + 1] # Extrai a subexpressão a ser guardada, usando i + 1 para incluir o parêntese de fechamento ')'
            legenda = remover_parenteses_externos(sub) # Rempove os parênyeses externos da subexpressão específica 

            # Evita duplicação de subexpressões 
            if legenda not in coluna_legenda:
                coluna_legenda.append(legenda) # Adiciona à lista "coluna_legenda"
                encontrar_relacoes(sub,coluna_legenda) # Procura subexpressões dentro dela de forma recursiva


def numero_variaveis_linhas(coluna_legenda):
    # Conta variáveis puras e define tamanho da tabela
    variaveis = 0

    for elemento in coluna_legenda:
        if len(elemento) == 1 and elemento.isalpha():
            variaveis += 1

    return variaveis, 2 ** variaveis


def gerar_combinacoes_bool(n, tabela):
    # gera todas combinações possíveis de True/False
    for i in range(2 ** n):
        linha = []

        for j in range(n - 1, -1, -1):
            linha.append(bool((i >> j) & 1))

        tabela.append(linha)

    return tabela


def gerar_bool_subexpressões(coluna_legenda,tabela):


    variaveis = [x for x in coluna_legenda
             if len(x)==1 and x.isalpha()]


    idx = {}

    for i,nome in enumerate(coluna_legenda):
        idx[nome]=i



    for linha in tabela[1:]:

        while len(linha)<len(coluna_legenda):
            linha.append(None)



    for i,linha in enumerate(tabela):

        if i==0:
            continue


        valores = {}

        for v in variaveis:
            valores[v]=linha[idx[v]]



        for coluna in coluna_legenda:


            if len(coluna)==1:
                continue


            linha[idx[coluna]] = avaliar(coluna,valores)



    return tabela

def programa():
    # ============================================================
    # ENTRADA
    # ============================================================
    expressao = input("Digite a expressão lógica: ").replace(" ", "")

    # validação de caracteres
    contador_invalidos=0
    for c in expressao:
        caracteres= [
        'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M',
        'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z',
        'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm',
        'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z','(',')','&','|','<','>','-','~'
    ]
        if c not in caracteres:
            contador_invalidos+=1
            break
    if contador_invalidos==1:
        print('Expressão Inválida (Presença de caracteres indevidos)')
    else:
        # validação de parênteses
        pilha = []
        erro = False

        for c in expressao:
            if c == "(":
                pilha.append(c)
            elif c == ")":
                if not pilha:
                    erro = True
                    break
                pilha.pop()

        if erro or pilha:
            print("Erro: parênteses inválidos.")

        else:

            # garante expressão bem formada
            if expressao[0] != "(" or expressao[-1] != ")":
                expressao = "(" + expressao + ")"

            expressao = remover_parenteses_externos(expressao)

            # ========================================================
            # CONSTRUÇÃO DO CABEÇALHO (COLUNAS DA TABELA)
            # ========================================================
            coluna_legenda = []

            # variáveis puras
            for c in expressao:
                if c.isalpha() and c not in coluna_legenda:
                    coluna_legenda.append(c)

            # negações explícitas
            for i, c in enumerate(expressao):
                if c.isalpha():
                    if i > 0 and expressao[i - 1] == "~":
                        token = f"~{c}"
                        if token not in coluna_legenda:
                            coluna_legenda.append(token)

            # subexpressões internas
            encontrar_relacoes(expressao,coluna_legenda)

            # expressão principal
            if expressao not in coluna_legenda:
                coluna_legenda.append(expressao)
            # remover duplicatas
            coluna_legenda = list(dict.fromkeys(coluna_legenda))

            # ==========================
            # ORDENAÇÃO DAS VARIÁVEIS
            # ==========================

            variaveis_ordenadas = sorted(
                [x for x in coluna_legenda if len(x) == 1]
            )

            outras_colunas = [
                x for x in coluna_legenda
                if len(x) != 1
                ]

            coluna_legenda = variaveis_ordenadas + outras_colunas

            # ============================================================
            # GERAÇÃO DA TABELA VERDADE E CLASSIFICAÇÃO AUTOMÁTICA
            # ============================================================
            variaveis, _ = numero_variaveis_linhas(coluna_legenda)

            tabela = [coluna_legenda]
            tabela = gerar_combinacoes_bool(variaveis, tabela)
            tabela = gerar_bool_subexpressões(coluna_legenda, tabela)

            true = 0

            for linha in tabela[1:]:
                if linha[-1]:
                    true += 1
            if true == len(tabela[1:]):
                print('A expressão lógica é uma Tautologia')
            elif true == 0:
                print('A expressão lógica é uma Contradição')
            else:
                print('A expressão lógica é uma Contingência')
            


            df = pd.DataFrame(tabela[1:], columns=tabela[0])
            display(df)
            return df
def comparar_tabelas():
    tabela1=programa()
    tabela2=programa()
    if len(tabela1) == len(tabela2):
        if tabela1.iloc[:, -1].equals(tabela2.iloc[:, -1]):
            print('As expressões são equivalentes')
        else:
            print('Não são equivalentes')
    else:
        print('Não são equivalentes')
def escolher():
    opção=input('O que você quer fazer?(Digite 1 para gerar 1 tabela verdade, Digite 2 para comparar duas tabelas)')
    if opção == '1':
        programa() 
    elif opção == '2':
        comparar_tabelas()
    else:
        print('Selecione uma opçaõ válida')
        escolher()      
escolher()



                